In [1]:
import numpy as np
from scipy import stats
import weightedstats as ws

# EDA

### Estimate Location

#### Mean
$$\frac{\sum_{i}^{n} x_{i}}{n}$$

#### Trimed Mean
$$\frac{\sum_{i=p+1}^{n - p} x_{i}}{n - 2p}$$

#### Weighted Mean
$$\frac{\sum_{i}^{n} w_{i}x_{i}}{\sum_{i}^{n} w_{i}}$$

#### Median
$$\text{if odd: }\frac{n + 1}{2}$$
$$\text{if even: }\frac{\frac{n}{2} + \frac{n + 1}{2}}{2}$$

#### Weighted Median
step 1: Calulate the sum of the weights
$$W = \sum_{i}^{n}w_i$$

step 2: Divide that value by two
$$Y = \frac{W}{2}$$

step 3: The weighted median is the value at this index where the associated cumulative weights is >= Y


### Estimate Variability (Dispersion)

#### Mean Deviation
Definition: The difference between the observed values and the estimate of location \
Synonyms: error, residuals \
Formula:

$$d_i = x_i - \bar{x} \quad \text{for } i = 1, 2, 3...n$$

#### Mean Absolute Deviation
Definition: The mean of the absolute values of the deviation (need to be absolute otherwise the mean of the deviation is always equal to 0)\
Synonyms: l1 Norm, Manhattan Norm\
Formula:
$$\frac{\sum_{i}^{n}|x_i + x|}{n}$$

#### Median Deviation
$$d_i = x_i - \tilde{x} \quad \text{for } i = 1, 2, 3...n$$

#### Median Absolute Deviation
$$MAD = median(|\mathbf{x} - \tilde{x}\mathbf{1}|)$$

#### Variance
Definition: The sum of squared deviation from the mean divided by n - 1 where n is the number of data values \
Synonym: mean-squared-error (MSE) \
Formula:
$$v^{2} = \frac{\sum{(x - \bar{x})^{2}}}{n - 1}$$
(just n works too)

#### Standard Deviation
Definition: Squared root of the Variance \
Synonym: l2-norm, Euclidian norm \
Formula:
$$v = \sqrt{v^{2}}$$

#### Range
$$max(x) - min(x)$$

#### Percentile
$$i = \frac{p}{100} * (N - 1)$$

#### IQR: Iter Quartile Range
$$IQR = Q3 - Q1$$

In [47]:
class tinyStatistician:
	def __init__(self):
		pass

	def mean(self, x: list[float]) -> float:
		if len(x) == 0:
			raise ValueError("List should have at least one value")
			
		return sum(x) / len(x)

	def trimmed_mean(self, x: list[float], p: float) -> float:
		if len(x) == 0:
			raise ValueError("List should have at least one value")
		if p < 0 or p >= 0.5:
			raise ValueError("trimmed value should be between 0 and 0.5")

		x_sort = sorted(x)

		p_index = int(len(x_sort) * p)

		x_trimmed = x_sort[p_index:len(x_sort) - p_index:]

		return sum(x_trimmed) / len(x_trimmed)

	def weighted_mean(self, x: list[float], w: list[float]) -> float:
		if len(x) == 0:
			raise ValueError("List should have at least one value")
		if len(x) != len(w):
			raise ValueError("Vector of Values and Wieghts should be of equal sizes")

		x_w = [x_ * w_ for x_, w_ in zip(x, w)]

		return sum(x_w) / sum(w)

	def median(self, x: list[float]) -> float:
		if len(x) == 0:
			raise ValueError("List should have at least one value")
			
		x = sorted(x)

		fiftieth = x[int(len(x) / 2)]
		fiftyfirst = x[int(((len(x) + 1) / 2) - 1)] # extra -1 in CS cause index starts at 0

		if len(x) % 2 == 0:
			return (fiftieth + fiftyfirst) / 2
		else:
			return fiftyfirst
		
	def weighted_median(self, x: list[float], w: list[float]) -> float:
		if len(x) == 0:
			raise ValueError("List should have at least one value")
		if len(x) != len(w):
			raise ValueError("Vector of Values and Wieghts should be of equal sizes")

		# Sort values and weights in the same order
		x_sorted, w_sorted = zip(*sorted(zip(x, w)))

		# Calculate sum of weights
		w_sum = sum(w)

		fifty_treshold = w_sum / 2

		cumulative_weight = 0
		for value, weigth in zip(x_sorted, w_sorted):
			cumulative_weight += weigth

			if cumulative_weight >= fifty_treshold:
				return value

	def mean_deviation(self, x: list[float]) -> list[float]:
		if len(x) == 0:
			raise ValueError("List should have at least one value")

		x_mean = self.mean(x)

		return [x_i - x_mean for x_i in x]

	def mean_absolute_deviation(self, x: list[float]) -> list[float]:
		if len(x) == 0:
			raise ValueError("List should have at least one value")

		x_dev = self.mean_deviation(x)
		x_abs_dev = [abs(x_i) for x_i in x_dev]

		return self.mean(x_abs_dev)

	def median_deviation(self, x: list[float]) -> list[float]:
		if len(x) == 0:
			raise ValueError("List should have at least one value")

		x_median = self.median(x)

		return [x_i - x_median for x_i in x]

	def median_absolute_deviation(self, x: list[float]) -> list[float]:
		if len(x) == 0:
			raise ValueError("List should have at least one value")

		x_dev = self.median_deviation(x)
		x_abs_dev = [abs(x_i) for x_i in x_dev]

		return self.median(x_abs_dev)

	def variance(self, x: list[float]) -> float:
		if len(x) == 0:
			raise ValueError("List should have at least one value")

		x_dev = self.mean_deviation(x)
		x_dev_sqr = sum(x_i ** 2 for x_i in x_dev)

		return x_dev_sqr / len(x)

	def standard_deviation(self, x: list[float]) -> float:
		if len(x) == 0:
			raise ValueError("List should have at least one value")

		return self.variance(x) ** 0.5

	def range(self, x: list[float]) -> float:
		if len(x) == 0:
			raise ValueError("List should have at least one value")
		return max(x) - min(x)

	def percentile(self, x: list[float], p: int) -> float:
		if len(x) == 0:
			raise ValueError("List should have at least one value")
		if p < 0 or p > 100:
			raise ValueError("Percitile should be between 0 and 100")

		x_sort = sorted(x)

		decimale = float(p / 100)

		p_index = int(len(x_sort) * decimale)

		return x_sort[p_index]

	def iqr(self, x: list[float]) -> float:
		if len(x) == 0:
			raise ValueError("List should have at least one value")

		return self.percentile(x, 75) - self.percentile(x, 25)
	

tstat = tinyStatistician()
a = [6, 10, 2, 35, 4, 12]

# TEST

print(f'tstat.mean() = {tstat.mean(a)}')
print(f'np.mean() = {np.mean(a)}')


p = 0.1
print(f'tstat.trimmed_mean() = {tstat.trimmed_mean(a, p)}')
print(f'np.trimmed_mean() = {stats.trim_mean(a, p)}\n')

w = [0.1, 0.5, 0.6, 0.99, 0, 1]
print(f'tstat.weighted_mean() = {tstat.weighted_mean(a, w)}')
print(f'np.weighted_mean() = {np.average(a, weights=w)}\n')

x_even = [6, 10, 2, 35, 4, 12]
x_odd = [6, 10, 2, 35, 4, 12, 5]
print(f'tstat.median(even) = {tstat.median(x_even)}')
print(f'np.median(even) = {np.median(x_even)}')
print(f'tstat.median(odd) = {tstat.median(x_odd)}')
print(f'np.median(odd) = {np.median(x_odd)}\n')


w_even = [0.1, 0.5, 0.6, 0.99, 0, 1]
w_odd= [0.1, 0.5, 0.6, 0.99, 0, 1, 0.753]
print(f'tstat.weighed_median(even) = {tstat.weighted_median(x_even, w_even)}')
print(f'ws.weighed_median(even) = {ws.weighted_median(x_even, w_even)}')
print(f'tstat.weighed_median(odd) = {tstat.weighted_median(x_odd, w_odd)}')
print(f'ws.weighed_median(odd) = {ws.weighted_median(x_odd, w_odd)}\n')

np_deviation = np.array(a) - np.mean(a)
print(f'tstat.mean_deviation() = {tstat.mean_deviation(a)}')
print(f'np.mean_deviation() = {np_deviation}')
np_abs_deviation = abs(np_deviation)
print(f'tstat.mean_absolute_deviation() = {tstat.mean_absolute_deviation(a)}')
print(f'np.mean_absolute_deviation() = {np.mean(np_abs_deviation)}\n')

np_median_deviation = np.array(a) - np.median(a)
print(f'tstat.median_deviation() = {tstat.median_deviation(a)}')
print(f'np.median_deviation() = {np_median_deviation}\n')

np_abs_deviation_median = abs(np_median_deviation)
print(f'tstat.mad() = {tstat.median_absolute_deviation(a)}')
print(f'np.mad() = {np.median(np_abs_deviation_median)}\n')


print(f'tstat.variance() = {tstat.variance(a)}')
print(f'np.variance() = {np.var(a)}')
print(f'tstat.std() = {tstat.standard_deviation(a)}')
print(f'np.std() = {np.std(a)}\n')

print(f'tstat.range() = {tstat.range(a)}')
print(f'np.range() = {np.ptp(a)}\n')

p25, p50, p75 = np.percentile(a, [25, 50, 75])
print(f'tstat.percentile(25) = {tstat.percentile(a, 25)}')
print(f'np.percentile(25) = {p25}')
print(f'tstat.percentile(50) = {tstat.percentile(a, 50)}')
print(f'np.percentile(50) = {p50}')
print(f'tstat.percentile(75) = {tstat.percentile(a, 75)}')
print(f'np.percentile(75) = {p75}\n')

print(f'tstat.iqr() = {tstat.iqr(a)}')
print(f'scipy.iqr() = {stats.iqr(a)}\n')

tstat.mean() = 11.5
np.mean() = 11.5
tstat.trimmed_mean() = 11.5
np.trimmed_mean() = 11.5

tstat.weighted_mean() = 16.755485893416928
np.weighted_mean() = 16.755485893416928

tstat.median(even) = 8.0
np.median(even) = 8.0
tstat.median(odd) = 6
np.median(odd) = 6.0

tstat.weighed_median(even) = 12
ws.weighed_median(even) = 12
tstat.weighed_median(odd) = 12
ws.weighed_median(odd) = 12

tstat.mean_deviation() = [-5.5, -1.5, -9.5, 23.5, -7.5, 0.5]
np.mean_deviation() = [-5.5 -1.5 -9.5 23.5 -7.5  0.5]
tstat.mean_absolute_deviation() = 8.0
np.mean_absolute_deviation() = 8.0

tstat.median_deviation() = [-2.0, 2.0, -6.0, 27.0, -4.0, 4.0]
np.median_deviation() = [-2.  2. -6. 27. -4.  4.]

tstat.mad() = 4.0
np.mad() = 4.0

tstat.variance() = 121.91666666666667
np.variance() = 121.91666666666667
tstat.std() = 11.04158805003459
np.std() = 11.04158805003459

tstat.range() = 33
np.range() = 33

tstat.percentile(25) = 4
np.percentile(25) = 4.5
tstat.percentile(50) = 10
np.percentile(50) = 8.0
tstat.p